# Extractor de Costeo Unitario desde PDFs — Claude (lectura nativa)

**Documentación metodológica del script de extracción de costeo unitario**

Equipo de Costeo PDET — ART

---

## Objetivo del documento

Este notebook explica, fase por fase, el script que lee cada PDF (Estudio
Previo, Anexo Técnico o similar) descargado de SECOP-II y extrae **cantidad +
unidad de medida + precio total**, calculando
`costo_unitario_cop = precio_cop / cantidad` (costo por km, por vivienda, por
centro de salud, etc.), junto con componentes del costo y campos de
auditoría. Esta versión usa **Claude (Anthropic)** como proveedor.

> Esta versión ya no está pensada solo para el trío piloto de indicadores:
> corre sobre **el conjunto pleno de indicadores** del proyecto — cualquier
> carpeta de PDFs organizada por `{codigo_indicador}/{codigo_contrato}.pdf`
> sirve de entrada, sin importar cuántos indicadores contenga.

## ⚠ Diferencia importante entre proveedores para esta tarea

Esta tarea depende mucho de leer **tablas** (presupuestos, APUs, cantidades
de obra) dentro del PDF, a veces con estructura visual compleja o el
documento escaneado como imagen.

- **Claude (Anthropic)** lee el PDF de forma **nativa** (páginas como imagen
  + texto), incluyendo tablas y documentos escaneados. Es el camino de mejor
  fidelidad para esta tarea — por eso es el modelo recomendado para todo el
  equipo (**Claude Sonnet 4.6**).
- **OpenAI / DeepSeek / custom** (vía la capa `OpenAICompatibleProvider` de
  este mismo script) reciben el **texto extraído** del PDF con `pypdf`, no
  las imágenes de las páginas. Funciona bien en PDFs "de texto" normales,
  pero **pierde la estructura de tablas complejas** y **no funciona en
  absoluto si el PDF es un escaneo** (imagen sin texto seleccionable) — en
  ese caso el documento llega vacío al modelo y la extracción sale con
  confianza "baja" o nula.
  > Si un PDF es un escaneo y te tocó un proveedor sin lectura nativa, repórtalo
  > para que alguien con Claude lo corra, o usa la variante con OCR local
  > (ver el notebook del extractor con DeepSeek).

## Campos nuevos respecto al script original (`extraer_costeo_pdfs_F_v2_FINAL.py`)

- `componentes_costo`: desglose de componentes del costo si el documento lo
  reporta (materiales, mano de obra, AIU, transporte, interventoría...).
- `fuente_cantidad`: dónde encontró la cantidad dentro del documento (tabla,
  página, ítem) — para poder auditar la extracción después.

> **Generalización de la base de entrada**
>
> Entrada: una carpeta con PDFs nombrados como el **código del contrato**
> (ej. `CO1.PCCNTR.5771514.pdf`), organizados en subcarpetas por indicador —
> tal como los deja el descargador de PDFs en
> `Descargas/{codigo_indicador}/`. Se puede apuntar directo a
> `Descargas/P5.25` o a `Descargas/` completo (busca recursivo en
> subcarpetas, para **todos** los indicadores a la vez). No hay ninguna
> dependencia de nombres de indicador específicos: el código del indicador
> se infiere del nombre de la subcarpeta.

## Panorama general del pipeline

```
Carpeta de PDFs (Descargas/{codigo_indicador}/{codigo_contrato}[.pdf|_AT.pdf])
        │
        ▼
Carga de resultados previos (reanudación — se omiten PDFs ya procesados)
        │
        ▼
Por cada PDF pendiente:
  validación (tamaño/existencia) ──► envío al proveedor (Claude: PDF nativo)
        │
        ▼
  Parseo de la respuesta JSON (con reintentos ante error/rate-limit)
        │
        ▼
  Cálculo de costo_unitario_cop si el modelo no lo entregó
        │
        ▼
Checkpoint cada N PDFs + Excel final con formato condicional por confianza
```

## Control global de warnings y errores

Misma celda reutilizable de los notebooks anteriores del pipeline: modificable por celda, ya sea cambiando las variables globales o sobreescribiendo `warnings.filterwarnings(...)` puntualmente.

In [ ]:
# ── Control global de warnings y errores (modificable por celda) ────────
import warnings

MOSTRAR_WARNINGS = False   # -> True para ver warnings de pandas/openpyxl aquí
DETENER_EN_ERROR = False   # -> True para propagar errores inesperados en vez de solo loguearlos

if MOSTRAR_WARNINGS:
    warnings.filterwarnings("default")
else:
    warnings.filterwarnings("ignore")

# Para reactivar warnings SOLO en una celda puntual (sin afectar el resto):
#   with warnings.catch_warnings():
#       warnings.filterwarnings("default")
#       ... código a depurar ...

## Configuración global

Toda la configuración editable vive en las siguientes celdas — **un solo modelo por persona**, editado una vez aquí; el resto del notebook no cambia.

In [ ]:
import os
import re
import sys
import json
import time
import base64
import logging
from pathlib import Path
from datetime import datetime
from typing import Optional

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

In [ ]:
# Opción recomendada para todo el equipo: Claude Sonnet 4.6 — es el único
# camino con lectura NATIVA del PDF (ver aviso arriba sobre tablas/escaneos).
PROVEEDOR = "claude"
MODELO    = "claude-sonnet-4-6"

# Extended thinking (SOLO aplica a Claude). 0 = desactivado (recomendado
# para Sonnet en esta tarea). Si usas Opus y quieres razonamiento extendido,
# pon un budget > 0 y sube MAX_TOKENS_RESPUESTA por encima de ese budget.
THINKING_BUDGET_TOKENS = 0

# ── Si a ti te tocó OTRO modelo, comenta las 2 líneas de arriba y
#    descomenta EL BLOQUE que corresponda (nunca dejes dos activos).
#    Recuerda: estos leen TEXTO extraído del PDF, no el PDF nativo.

# --- OpenAI ----------------------------------------------------------
# PROVEEDOR = "openai"
# MODELO    = "gpt-5"                # confirmar en platform.openai.com/docs

# --- DeepSeek (sin OCR) ------------------------------------------------
# PROVEEDOR = "deepseek"
# MODELO    = "deepseek-chat"        # ver notebook aparte con OCR local

# --- Cualquier otro endpoint compatible con OpenAI ----------------------
# PROVEEDOR   = "custom"
# MODELO      = "llama-3.3-70b"
# BASE_URL    = "https://api.groq.com/openai/v1"
# API_KEY_ENV = "CUSTOM_API_KEY"

BASE_URL    = None
API_KEY_ENV = None

In [ ]:
# ── ENTRADA (GENERAL) ─────────────────────────────────────────────────────
# Carpeta con PDFs organizados en subcarpetas por indicador:
#   Descargas/{codigo_indicador}/{codigo_contrato}.pdf
# Apunta aquí a la carpeta raíz completa para procesar TODOS los indicadores,
# o a una subcarpeta puntual para procesar solo uno.
RUTA_PDFS_DEFAULT = "Descargas"
EXCEL_SALIDA      = "costeo_unitario.xlsx"
LOG_FILE          = "extraccion_costeo.log"

MAX_TOKENS_RESPUESTA  = 2000 if THINKING_BUDGET_TOKENS == 0 else max(8000, THINKING_BUDGET_TOKENS + 3000)
MAX_TAMANO_MB         = 30
PAUSA_ENTRE_LLAMADAS  = 5
MAX_REINTENTOS        = 5
PAUSA_RATE_LIMIT      = 90
CHECKPOINT_CADA       = 5
LIMITE_CHARS_TEXTO_PDF = 120_000   # tope de texto enviado a proveedores sin lectura nativa

### Logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

## Fase 1 — Prompt de extracción

El prompt le pide al modelo 15 campos, todos con instrucción explícita de
**no inventar** (`null` si el dato no aparece). Los dos campos marcados como
⚠️ CRÍTICOS (`cantidad` y `unidad_cantidad`) son los que sostienen todo el
cálculo de costeo unitario, así que el prompt les dedica la instrucción más
detallada: dónde buscar (objeto, alcance, anexo técnico, APUs, tablas de
cantidades) y varios ejemplos concretos de unidades PDET (hectáreas, km,
familias, viviendas, centros de salud, cupos de formación...).

In [ ]:
PROMPT_TEMPLATE = """Eres un analista experto en contratación pública colombiana (SECOP-II). \
Tu tarea es leer el documento contractual adjunto (Estudio Previo, Anexo Técnico o \
similar) y extraer la información clave para calcular el COSTEO UNITARIO del contrato.

CONTEXTO CRÍTICO — CÓDIGO DEL CONTRATO
──────────────────────────────────────
El nombre del archivo PDF que estás analizando corresponde al CÓDIGO DEL CONTRATO en SECOP-II:

    CÓDIGO_CONTRATO_ARCHIVO: {codigo_archivo}

Usa este código como referencia principal de identificación. Si dentro del documento aparecen \
identificadores adicionales (número interno tipo CCFV-065-2024, número de proceso de selección \
tipo LP-001-2024, etc.), inclúyelos en el campo `codigo_contrato` concatenándolos con " | " al \
código anterior.

CAMPOS A EXTRAER
────────────────
Extrae los siguientes campos. Si un dato NO aparece explícitamente en el documento, \
pon `null` (NO inventes valores).

1. **codigo_contrato**: identificadores del contrato encontrados EN el documento (número \
interno, número de proceso, etc.). Si aparecen varios, concaténalos con " | ". null si no \
encuentras ninguno en el texto.

2. **descripcion_contrato**: objeto del contrato tal como aparece en el documento \
(texto completo del objeto, máximo 500 caracteres).

3. **municipio**: municipio(s) o lugar(es) específicos de ejecución del contrato. \
Si son varios, sepáralos con "; ". Usa los nombres exactos como aparecen en el documento.

4. **departamento**: departamento(s) correspondiente(s) (Colombia). Separa con "; " si son varios.

5. **subregion_impacto**: si el documento menciona explícitamente una subregión PDET \
(ej: "Alto Patía - Norte del Cauca", "Catatumbo", "Sur de Bolívar", "Macarena - Guaviare") \
o una región de impacto específica, indícala. null si no aparece.

6. **year_contrato**: año del contrato (formato YYYY). Si aparece fecha completa, extrae \
el año. Si hay varias fechas (firma, inicio, terminación), usa la de FIRMA o SUSCRIPCIÓN. \
null si no aparece.

7. **precio_cop**: valor total del contrato en pesos colombianos (COP), como número entero \
sin puntos, comas ni símbolo $. Ejemplo: 7035424595. Si el valor está en otra moneda, \
extráelo tal cual y anota la moneda en `moneda`.

8. **moneda**: "COP", "USD", "EUR", etc. Por defecto "COP".

9. **cantidad**: ⚠️ CAMPO CRÍTICO — cantidad física que el contrato entrega/ejecuta, en su \
unidad natural. Busca con detenimiento en el objeto, alcance, obligaciones específicas, \
anexo técnico, presupuesto detallado, APUs y tablas de cantidades. \
Ejemplos: 683600 (hectáreas), 12.5 (km de vía), 240 (familias beneficiarias), \
177 (viviendas mejoradas), 4 (infraestructuras de salud: 1 centro + 3 puestos), \
150 (cupos de formación), 35 (organizaciones apoyadas), 8 (kits entregados). \
Usa número decimal con punto. null SOLO si después de revisar el documento completo \
realmente no aparece ninguna cantidad física asociada al producto/servicio.

10. **unidad_cantidad**: ⚠️ CAMPO CRÍTICO — unidad física de la cantidad (ej: "hectárea", \
"km", "familia", "vivienda", "vivienda mejorada", "centro de salud", "cupo de formación", \
"estudiante", "kit entregado", "metro lineal"). Debe ser coherente con `cantidad`. null si \
no aparece.

11. **costo_unitario_cop**: costo por unidad en COP. Si el documento lo reporta explícitamente \
(p. ej. en el APU o presupuesto), úsalo. Si no, calcúlalo como `precio_cop / cantidad`. \
Redondea a entero. null si no se puede calcular (faltan datos).

12. **componentes_costo**: desglose de los componentes del costo si el documento lo reporta \
— por ejemplo materiales, mano de obra, AIU (Administración, Imprevistos, Utilidad), \
transporte, interventoría, dotación. Texto breve tipo "Materiales 45%, Mano de obra 30%, \
AIU 15%, Transporte 10%" o una lista de los ítems principales del presupuesto/APU con su \
peso aproximado si el documento lo permite. null si el documento no desglosa el costo.

13. **fuente_cantidad**: nota breve de DÓNDE dentro del documento encontraste la cantidad \
— por ejemplo "Tabla de cantidades de obra, pág. 14", "Anexo técnico, ítem 3", "Cláusula \
segunda - objeto", "Presupuesto detallado, fila 'Vivienda mejorada'". Esto es para poder \
auditar la extracción después. null si no aplica.

14. **confianza**: tu confianza en la extracción — "alta", "media" o "baja". \
"baja" si el documento es ambiguo, está escaneado con mala resolución, o no encontraste \
la mayoría de los campos clave (en especial `cantidad` y `precio_cop`).

15. **observaciones**: nota del analista (máximo 300 caracteres). Por ejemplo: \
"Contrato incluye interventoría dentro del valor"; "Cantidad aproximada"; \
"El documento es una adición, no el contrato original"; "No especifica cantidad, solo valor total"; \
"Cantidad inferida del anexo técnico tabla 4".

FORMATO DE RESPUESTA
────────────────────
Responde ÚNICAMENTE con un objeto JSON válido. Sin texto antes ni después, sin backticks, \
sin explicaciones:

{{
  "codigo_contrato": "...",
  "descripcion_contrato": "...",
  "municipio": "...",
  "departamento": "...",
  "subregion_impacto": "...",
  "year_contrato": 2024,
  "precio_cop": 0,
  "moneda": "COP",
  "cantidad": 0.0,
  "unidad_cantidad": "...",
  "costo_unitario_cop": 0,
  "componentes_costo": "...",
  "fuente_cantidad": "...",
  "confianza": "alta",
  "observaciones": "..."
}}
"""

## Fase 2 — Capa de proveedores

In [ ]:
class BaseProvider:
    label = "base"
    supports_native_pdf = False

    def complete_pdf(self, path_pdf: Path, prompt: str, max_tokens: int) -> str:
        raise NotImplementedError


class AnthropicProvider(BaseProvider):
    label = "claude"
    supports_native_pdf = True

    def __init__(self, model, thinking_budget=0):
        import anthropic
        key = os.environ.get("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("Falta ANTHROPIC_API_KEY en el entorno.")
        self._anthropic = anthropic
        self.client = anthropic.Anthropic(api_key=key)
        self.model = model
        self.thinking_budget = thinking_budget

    def complete_pdf(self, path_pdf, prompt, max_tokens):
        with open(path_pdf, "rb") as f:
            pdf_b64 = base64.standard_b64encode(f.read()).decode("utf-8")

        contenido = [
            {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64}},
            {"type": "text", "text": prompt},
        ]
        kwargs = dict(model=self.model, max_tokens=max_tokens, messages=[{"role": "user", "content": contenido}])
        if self.thinking_budget and self.thinking_budget > 0:
            kwargs["thinking"] = {"type": "enabled", "budget_tokens": self.thinking_budget}

        msg = self.client.messages.create(**kwargs)
        if hasattr(msg, "usage"):
            log.info(f"    tokens in={msg.usage.input_tokens} out={msg.usage.output_tokens}")

        raw_parts = [getattr(blk, "text", "") or "" for blk in msg.content if getattr(blk, "type", None) == "text"]
        return "\n".join(raw_parts).strip()


class OpenAICompatibleProvider(BaseProvider):
    """
    OpenAI, DeepSeek, o cualquier otro endpoint Chat-Completions.
    NO lee el PDF de forma nativa: extrae el texto con pypdf y lo manda
    como texto plano. Ver aviso arriba sobre tablas/escaneos.
    """
    supports_native_pdf = False

    def __init__(self, label, model, api_key_env, base_url=None):
        import openai
        key = os.environ.get(api_key_env)
        if not key:
            raise RuntimeError(f"Falta {api_key_env} en el entorno.")
        self.label = label
        self.client = openai.OpenAI(api_key=key, base_url=base_url)
        self.model = model

    def complete_pdf(self, path_pdf, prompt, max_tokens):
        texto = extraer_texto_pdf(path_pdf)
        if not texto.strip():
            raise RuntimeError("El PDF no tiene texto seleccionable (parece un escaneo) — "
                                "este proveedor no puede leerlo sin OCR previo.")
        prompt_full = (
            prompt
            + "\n\nTEXTO EXTRAÍDO DEL PDF (sin imágenes ni estructura de tabla original — "
              "interpreta con cautela cualquier tabla que se vea desalineada):\n"
            + texto[:LIMITE_CHARS_TEXTO_PDF]
        )
        resp = self.client.chat.completions.create(
            model=self.model, max_tokens=max_tokens, temperature=0,
            messages=[{"role": "user", "content": prompt_full}],
        )
        return resp.choices[0].message.content


def extraer_texto_pdf(path_pdf: Path) -> str:
    from pypdf import PdfReader
    try:
        reader = PdfReader(str(path_pdf))
    except Exception as e:
        log.warning(f"  No se pudo abrir el PDF con pypdf: {e}")
        return ""
    partes = []
    for page in reader.pages:
        try:
            partes.append(page.extract_text() or "")
        except Exception:
            continue
    return "\n".join(partes)


def build_provider():
    if PROVEEDOR == "claude":
        return AnthropicProvider(model=MODELO, thinking_budget=THINKING_BUDGET_TOKENS)
    if PROVEEDOR == "openai":
        return OpenAICompatibleProvider("openai", MODELO, "OPENAI_API_KEY")
    if PROVEEDOR == "deepseek":
        return OpenAICompatibleProvider("deepseek", MODELO, "DEEPSEEK_API_KEY", base_url="https://api.deepseek.com")
    if PROVEEDOR == "custom":
        if not BASE_URL:
            raise RuntimeError("PROVEEDOR='custom' requiere BASE_URL en la CONFIGURACIÓN.")
        return OpenAICompatibleProvider("custom", MODELO, API_KEY_ENV or "CUSTOM_API_KEY", base_url=BASE_URL)
    raise RuntimeError(f"PROVEEDOR desconocido: '{PROVEEDOR}'")

`AnthropicProvider.complete_pdf` manda el PDF como **documento base64 nativo** (no texto extraído) — Claude procesa cada página como si la estuviera viendo, tablas y escaneos incluidos. `OpenAICompatibleProvider` es la ruta de compatibilidad para cualquier otro modelo, pero requiere que el PDF tenga texto seleccionable (ver el notebook del extractor con DeepSeek+OCR para el caso de documentos escaneados).

## Fase 3 — Utilidades de archivo y validación

In [ ]:
def pdf_apto(path_pdf: Path) -> tuple:
    if not path_pdf.exists():
        return False, "archivo no existe"
    size_mb = path_pdf.stat().st_size / (1024 * 1024)
    if size_mb > MAX_TAMANO_MB:
        return False, f"demasiado grande ({size_mb:.1f} MB > {MAX_TAMANO_MB} MB)"
    if size_mb < 0.001:
        return False, "archivo vacío"
    return True, f"{size_mb:.2f} MB"


def codigo_desde_nombre(path_pdf: Path) -> str:
    """El nombre del archivo SIN extensión ni sufijo _EP/_AT es el código de contrato."""
    stem = path_pdf.stem.strip()
    return re.sub(r'_(EP|AT|OT)(_\d+)?$', '', stem)


def indicador_desde_ruta(path_pdf: Path, ruta_raiz: Path) -> str:
    """Si el PDF está en Descargas/{codigo_indicador}/archivo.pdf, extrae el código del indicador."""
    try:
        rel = path_pdf.relative_to(ruta_raiz)
        if len(rel.parts) > 1:
            return rel.parts[0]
    except ValueError:
        pass
    return ""

`indicador_desde_ruta` es lo que permite correr el extractor sobre **toda** la carpeta `Descargas/` de una sola vez y aun así saber a qué indicador pertenece cada PDF: el código del indicador es simplemente el nombre de la subcarpeta inmediata, sin necesidad de mantener ninguna lista de indicadores dentro de este script.

## Fase 4 — Llamada al modelo con reintentos

In [ ]:
def llamar_modelo_pdf(provider, path_pdf: Path, codigo_archivo: str) -> Optional[dict]:
    prompt = PROMPT_TEMPLATE.format(codigo_archivo=codigo_archivo)

    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            raw = provider.complete_pdf(path_pdf, prompt, MAX_TOKENS_RESPUESTA)
            raw = raw.replace("```json", "").replace("```", "").strip()
            m = re.search(r"\{.*\}", raw, re.DOTALL)
            if m:
                raw = m.group(0)
            return json.loads(raw)

        except json.JSONDecodeError as e:
            log.warning(f"  JSON inválido (intento {intento}): {e}")
            time.sleep(2 * intento)

        except RuntimeError as e:
            # p.ej. "PDF sin texto seleccionable" en proveedores sin lectura nativa — no reintentar
            log.warning(f"  {e}")
            return None

        except Exception as e:
            msg = str(e).lower()
            if "rate" in msg or "429" in msg:
                log.warning(f"  Rate limit [{provider.label}] (intento {intento}), esperando {PAUSA_RATE_LIMIT}s...")
                time.sleep(PAUSA_RATE_LIMIT)
            elif "400" in msg:
                log.warning(f"  Error 400 [{provider.label}]: {e} — no se reintenta (PDF probablemente corrupto)")
                return None
            else:
                log.warning(f"  Error inesperado [{provider.label}] (intento {intento}): {type(e).__name__}: {e}")
                time.sleep(3 * intento)

    log.error(f"  Falla tras {MAX_REINTENTOS} intentos: {path_pdf.name}")
    return None

Cada tipo de falla se trata distinto, en vez de un `except Exception` genérico:

- **JSON inválido** → reintenta (a veces el modelo agrega texto extra alrededor del JSON).
- **`RuntimeError`** (p. ej. PDF sin texto seleccionable) → **no reintenta**: el problema es el archivo, no una falla transitoria de red.
- **Rate limit (429)** → espera `PAUSA_RATE_LIMIT` segundos (90s por defecto) antes de reintentar.
- **Error 400** → no reintenta; el PDF probablemente está corrupto.
- **Cualquier otro error** → espera con backoff creciente (`3 * intento`) y reintenta.

Si tras `MAX_REINTENTOS` (5) sigue fallando, la función devuelve `None` y **ese PDF simplemente no se guarda** — quedará pendiente para la próxima ejecución (ver Fase 5, reanudación), en vez de guardar una fila vacía o inventada.

## Fase 5 — Excel: formato, guardado y reanudación

In [ ]:
def formatear_excel(path_excel: Path):
    wb = load_workbook(path_excel)
    ws = wb.active

    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    border = Border(left=Side(style="thin", color="CCCCCC"), right=Side(style="thin", color="CCCCCC"),
                     top=Side(style="thin", color="CCCCCC"), bottom=Side(style="thin", color="CCCCCC"))
    align_header = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = align_header
        cell.border = border
    ws.row_dimensions[1].height = 36
    ws.freeze_panes = "A2"

    anchos = {
        "archivo_pdf": 32, "codigo_indicador": 14, "codigo_contrato_archivo": 26,
        "codigo_contrato": 22, "descripcion_contrato": 50, "municipio": 26,
        "departamento": 16, "subregion_impacto": 22, "year_contrato": 10,
        "precio_cop": 16, "moneda": 8, "cantidad": 11, "unidad_cantidad": 18,
        "costo_unitario_cop": 16, "componentes_costo": 40, "fuente_cantidad": 32,
        "confianza": 10, "observaciones": 38, "fecha_extraccion": 17, "proveedor_modelo": 20,
    }
    for i, col in enumerate(ws[1], start=1):
        ws.column_dimensions[get_column_letter(i)].width = anchos.get(col.value, 15)

    for i, col in enumerate(ws[1], start=1):
        if col.value in ("precio_cop", "costo_unitario_cop"):
            letra = get_column_letter(i)
            for cell in ws[letra][1:]:
                if cell.value is not None:
                    cell.number_format = '"$"#,##0'

    fills_conf = {"alta": PatternFill("solid", fgColor="E8F5E9"),
                  "media": PatternFill("solid", fgColor="FFF9C4"),
                  "baja": PatternFill("solid", fgColor="FFEBEE")}
    col_conf_idx = next((i for i, col in enumerate(ws[1], start=1) if col.value == "confianza"), None)
    if col_conf_idx:
        for row in ws.iter_rows(min_row=2):
            val = row[col_conf_idx - 1].value
            fill = fills_conf.get(str(val).lower() if val else None)
            if fill:
                for cell in row:
                    cell.fill = fill

    wb.save(path_excel)


ORDEN_COLUMNAS = [
    "archivo_pdf", "codigo_indicador", "codigo_contrato_archivo", "codigo_contrato",
    "descripcion_contrato", "municipio", "departamento", "subregion_impacto",
    "year_contrato", "precio_cop", "moneda", "cantidad", "unidad_cantidad",
    "costo_unitario_cop", "componentes_costo", "fuente_cantidad",
    "confianza", "observaciones", "proveedor_modelo", "fecha_extraccion",
]


def guardar_excel(resultados: list, path: Path):
    df = pd.DataFrame(resultados)
    for c in ORDEN_COLUMNAS:
        if c not in df.columns:
            df[c] = None
    df = df[ORDEN_COLUMNAS]
    df.to_excel(path, index=False, sheet_name="Costeo_Unitario")
    try:
        formatear_excel(path)
    except Exception as e:
        log.warning(f"No se pudo aplicar formato al Excel: {e}")


def cargar_resultados_previos(path: Path) -> tuple:
    if not path.exists():
        return [], set()
    try:
        df_prev = pd.read_excel(path, sheet_name="Costeo_Unitario")
        if df_prev.empty or "archivo_pdf" not in df_prev.columns:
            return [], set()
        df_prev = df_prev.where(pd.notna(df_prev), None)
        resultados_previos = df_prev.to_dict(orient="records")
        procesados = set(df_prev["archivo_pdf"].dropna().astype(str).tolist())
        return resultados_previos, procesados
    except Exception as e:
        log.warning(f"No se pudo leer el Excel previo ({path.name}): {e}")
        log.warning("Se empezará desde cero.")
        return [], set()

El color de fondo por **nivel de confianza** (verde/amarillo/rojo)
permite escanear visualmente qué extracciones necesitan revisión manual sin
leer fila por fila. `cargar_resultados_previos` es lo que hace el script
**reanudable**: si ya existe un Excel de una corrida anterior, se leen sus
filas y el conjunto de archivos ya procesados, para no volver a gastar
tokens en ellos (patrón consistente con el resto del pipeline del equipo).

## Fase 6 — Proceso principal

En el script original esto se maneja con `argparse` (`--ruta-pdfs`,
`--salida`, `--limite`, `--rehacer`). En el notebook, `ejecutar_extraccion`
recibe los mismos parámetros como argumentos de función.

In [ ]:
def ejecutar_extraccion(ruta_pdfs=RUTA_PDFS_DEFAULT, salida=EXCEL_SALIDA,
                         limite=None, rehacer=False):
    print("=" * 70)
    print(f"  Extracción de costeo unitario — proveedor: {PROVEEDOR} ({MODELO})")
    if PROVEEDOR != "claude":
        print("  ⚠ Este proveedor lee TEXTO extraído del PDF, no el PDF nativo.")
        print("    Ver aviso al inicio del notebook sobre tablas y documentos escaneados.")
    print("=" * 70)

    try:
        provider = build_provider()
    except RuntimeError as e:
        log.error(str(e))
        log.error('  En PowerShell: $env:ANTHROPIC_API_KEY = "sk-ant-..."  (u OPENAI_API_KEY / DEEPSEEK_API_KEY)')
        return

    ruta_raiz = Path(ruta_pdfs)
    if not ruta_raiz.exists():
        log.error(f"No existe la ruta de PDFs: {ruta_raiz}")
        return

    pdfs = sorted(set(ruta_raiz.rglob("*.pdf")))
    if not pdfs:
        log.error(f"No se encontraron PDFs en {ruta_raiz}")
        return

    excel_path = Path(salida)
    resultados, procesados = cargar_resultados_previos(excel_path)

    if procesados and not rehacer:
        pdfs_pendientes = [p for p in pdfs if p.name not in procesados]
        ya_procesados = len(pdfs) - len(pdfs_pendientes)
    else:
        pdfs_pendientes = pdfs
        ya_procesados = 0
        if rehacer:
            resultados, procesados = [], set()

    log.info("=" * 70)
    log.info(f"  Ruta de PDFs:    {ruta_raiz}")
    log.info(f"  PDFs detectados: {len(pdfs)}")
    if ya_procesados > 0:
        log.info(f"  Ya procesados:   {ya_procesados} (se omiten)")
        log.info(f"  Pendientes:      {len(pdfs_pendientes)}")
    log.info(f"  Proveedor:       {PROVEEDOR} | Modelo: {MODELO}")
    log.info(f"  Excel salida:    {salida}")
    log.info("=" * 70)

    if not pdfs_pendientes:
        log.info("Todos los PDFs ya están procesados. Nada por hacer (usa rehacer=True para reprocesar).")
        return

    pdfs = pdfs_pendientes
    if limite:
        pdfs = pdfs[:limite]
        log.info(f"Modo prueba: se procesarán solo {len(pdfs)} PDFs pendientes")

    total, exitos, fallos = len(pdfs), 0, 0

    for idx, pdf in enumerate(pdfs, start=1):
        codigo_archivo = codigo_desde_nombre(pdf)
        codigo_indicador = indicador_desde_ruta(pdf, ruta_raiz)
        log.info(f"\n[{idx}/{total}] {pdf.name}  (indicador={codigo_indicador or '?'}, contrato={codigo_archivo})")

        ok, info = pdf_apto(pdf)
        if not ok:
            log.warning(f"  Descartado: {info}")
            resultados.append({
                "archivo_pdf": pdf.name, "codigo_indicador": codigo_indicador,
                "codigo_contrato_archivo": codigo_archivo, "confianza": "baja",
                "observaciones": f"PDF descartado: {info}",
                "proveedor_modelo": f"{PROVEEDOR}:{MODELO}",
                "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            fallos += 1
            continue

        log.info(f"  Enviando a {provider.label} ({info})...")
        extraccion = llamar_modelo_pdf(provider, pdf, codigo_archivo)

        if extraccion is None:
            log.warning("  Sin respuesta utilizable — NO se guarda (se reintentará en próxima ejecución)")
            fallos += 1
        else:
            fila = {
                "archivo_pdf": pdf.name, "codigo_indicador": codigo_indicador,
                "codigo_contrato_archivo": codigo_archivo,
                "codigo_contrato": extraccion.get("codigo_contrato"),
                "descripcion_contrato": extraccion.get("descripcion_contrato"),
                "municipio": extraccion.get("municipio"),
                "departamento": extraccion.get("departamento"),
                "subregion_impacto": extraccion.get("subregion_impacto"),
                "year_contrato": extraccion.get("year_contrato"),
                "precio_cop": extraccion.get("precio_cop"),
                "moneda": extraccion.get("moneda"),
                "cantidad": extraccion.get("cantidad"),
                "unidad_cantidad": extraccion.get("unidad_cantidad"),
                "costo_unitario_cop": extraccion.get("costo_unitario_cop"),
                "componentes_costo": extraccion.get("componentes_costo"),
                "fuente_cantidad": extraccion.get("fuente_cantidad"),
                "confianza": extraccion.get("confianza"),
                "observaciones": extraccion.get("observaciones"),
                "proveedor_modelo": f"{PROVEEDOR}:{MODELO}",
                "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            }
            try:
                if (not fila["costo_unitario_cop"] and fila["precio_cop"] and fila["cantidad"]
                        and float(fila["cantidad"]) > 0):
                    fila["costo_unitario_cop"] = round(float(fila["precio_cop"]) / float(fila["cantidad"]))
            except (TypeError, ValueError):
                pass

            resultados.append(fila)
            exitos += 1
            log.info(
                f"  OK {fila.get('municipio') or '-'} | ${fila.get('precio_cop') or '-'} | "
                f"{fila.get('cantidad') or '-'} {fila.get('unidad_cantidad') or ''} | conf={fila.get('confianza')}"
            )

        time.sleep(PAUSA_ENTRE_LLAMADAS)

        if idx % CHECKPOINT_CADA == 0 or idx == total:
            guardar_excel(resultados, Path(salida))
            log.info(f"  [Checkpoint] {idx}/{total} guardados en {salida}")

    guardar_excel(resultados, Path(salida))

    log.info("\n" + "=" * 70)
    log.info("RESUMEN FINAL")
    log.info("=" * 70)
    log.info(f"  PDFs procesados:       {total}")
    log.info(f"  Extracciones exitosas: {exitos}")
    log.info(f"  Fallos:                {fallos}")

    df = pd.DataFrame(resultados)
    if not df.empty and "costo_unitario_cop" in df.columns:
        log.info(f"  Con costo unitario:    {df['costo_unitario_cop'].notna().sum()}")
        log.info(f"  Con cantidad:          {df['cantidad'].notna().sum()}")
        log.info(f"  Con precio:            {df['precio_cop'].notna().sum()}")
        for nivel in ["alta", "media", "baja"]:
            log.info(f"  Confianza '{nivel}': {(df['confianza'] == nivel).sum()}")

    log.info(f"\n  Archivo final: {Path(salida).resolve()}")
    log.info("=" * 70)

Puntos clave del flujo:

- **Reanudación por nombre de archivo** (`archivo_pdf` ya en el Excel previo) — correr el notebook dos veces no duplica ni regasta tokens en PDFs ya procesados.
- **Checkpoint cada 5 PDFs** (`CHECKPOINT_CADA`) — si el proceso se interrumpe a la mitad de un lote grande, no se pierde el trabajo ya hecho.
- El **cálculo de respaldo** de `costo_unitario_cop` (`precio_cop / cantidad`) solo se aplica si el modelo no lo entregó ya calculado — se confía primero en el valor que reporta el propio documento (ej. de un APU), y solo se calcula si hace falta.

## Fase 7 — Ejecución

Por defecto la celda queda con `limite=1` y apuntando a una ruta de prueba
para que el notebook se pueda ejecutar de principio a fin sin credenciales
ni PDFs reales (fallará de forma controlada si no encuentra la carpeta —
ver captura de errores abajo). Para una corrida real: ajustar `RUTA_PDFS`,
poner `LIMITE = None` (o un número para una prueba parcial) y asegurarse de
tener `ANTHROPIC_API_KEY` en el entorno.

In [ ]:
RUTA_PDFS = RUTA_PDFS_DEFAULT   # -> cambiar a la carpeta real de descargas
SALIDA    = EXCEL_SALIDA
LIMITE    = None                # -> un número para procesar solo N PDFs de prueba
REHACER   = False                # -> True para ignorar el Excel existente y reprocesar todo

try:
    ejecutar_extraccion(ruta_pdfs=RUTA_PDFS, salida=SALIDA, limite=LIMITE, rehacer=REHACER)
except Exception as e:
    log.error(f"Fallo en la ejecución: {e}")
    if DETENER_EN_ERROR:
        raise